# Baaki: Hindi / English / Hinglish Voice → Collection Plan

This notebook:
1. Records audio in the browser microphone.
2. Uses OpenAI Whisper (`openai-whisper`) for speech-to-text.
3. Extracts the requested rupee amount and deadline using a transparent parser.
4. Saves the request to CSV.
5. Loads a Baaki invoice dataset and produces a ranked collection shortlist.
6. Exports recommendations to CSV.

**Colab setup:** Runtime → Change runtime type → GPU is recommended, but CPU can work more slowly. Run cells from top to bottom. If Colab asks for microphone permission, allow it.

The amount/deadline extraction is a lightweight rule-based parser, not a separate trained NLP model. Review the transcript and extracted fields before relying on the recommendations.

## 1. Install dependencies
Run this cell first. It uses `openai-whisper`, not `faster-whisper`, so it does not use the PyAV `metadata_errors` code path that caused your earlier error.

In [ ]:
%pip -q install --upgrade pip
%pip -q install openai-whisper pandas==2.2.3 ipywidgets

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. Restart runtime if prompted
If Colab shows a restart prompt after installation, choose **Restart session**, then continue from the next cell. Do not rerun the install cell unless needed.

In [ ]:
import os
import re
import json
import subprocess
from pathlib import Path

import pandas as pd
import torch
import whisper

print("Python ready")
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Python ready
Torch: 2.11.0+cpu
CUDA available: False


## 3. Load Whisper speech recognition model
The `small` model is a practical balance for Colab. It can recognize Hindi and English; Hinglish quality may vary, so always review the transcript.

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WHISPER_MODEL_NAME = "small"

asr_model = whisper.load_model(WHISPER_MODEL_NAME, device=DEVICE)
print(f"Whisper '{WHISPER_MODEL_NAME}' loaded on {DEVICE}.")

Whisper 'small' loaded on cpu.


## 4. Record from your microphone

This uses the browser microphone through JavaScript and saves a WAV file in the Colab runtime. Run the cell, click **Start recording**, speak, then click **Stop recording**. Allow microphone access when asked.

In [ ]:
from IPython.display import display, Javascript, Audio
from google.colab import output
import base64
import wave
import io
import numpy as np

def record_audio_browser(filename="baaki_voice.wav", seconds=8):
    """Record from browser microphone and save a WAV file in Colab."""
    js = Javascript(r"""
    async function recordAudio(seconds) {
      const stream = await navigator.mediaDevices.getUserMedia({audio: true});
      const recorder = new MediaRecorder(stream);
      const chunks = [];
      recorder.ondataavailable = e => chunks.push(e.data);
      recorder.start();
      await new Promise(resolve => setTimeout(resolve, seconds * 1000));
      recorder.stop();
      await new Promise(resolve => recorder.onstop = resolve);
      stream.getTracks().forEach(track => track.stop());
      const blob = new Blob(chunks, {type: recorder.mimeType});
      const arrayBuffer = await blob.arrayBuffer();
      const bytes = new Uint8Array(arrayBuffer);
      let binary = '';
      const chunkSize = 0x8000;
      for (let i = 0; i < bytes.length; i += chunkSize) {
        binary += String.fromCharCode.apply(null, bytes.subarray(i, i + chunkSize));
      }
      return {mimeType: blob.type, data: btoa(binary)};
    }
    """)
    display(js)
    print(f"Recording for {seconds} seconds... speak clearly.")
    result = output.eval_js(f"recordAudio({int(seconds)})")
    raw = base64.b64decode(result["data"])
    # Browser MediaRecorder output is usually webm/opus. Save it, then use ffmpeg to make WAV.
    raw_path = "/content/baaki_recording.webm"
    with open(raw_path, "wb") as f:
        f.write(raw)
    converted = subprocess.run(
        ["ffmpeg", "-y", "-i", raw_path, "-ac", "1", "-ar", "16000", filename],
        capture_output=True, text=True
    )
    if converted.returncode != 0:
        raise RuntimeError("Audio conversion failed. ffmpeg output: " + converted.stderr[-1500:])
    print("Saved:", filename)
    return filename

# Change seconds to 10, 15, etc. if you need more speaking time.
audio_path = record_audio_browser(seconds=8)
display(Audio(audio_path))

<IPython.core.display.Javascript object>

Recording for 8 seconds... speak clearly.
Saved: baaki_voice.wav


## 5. Transcribe audio to text
This cell only does speech-to-text. It does not extract the amount or deadline yet.

In [ ]:
def transcribe_audio(audio_file, language=None):
    # language=None lets Whisper detect the language; explicit "hi" can help for Hindi-heavy speech.
    options = {"fp16": torch.cuda.is_available(), "verbose": False}
    if language:
        options["language"] = language
    result = asr_model.transcribe(audio_file, **options)
    return result["text"].strip(), result

transcript, transcription_result = transcribe_audio(audio_path)
print("TRANSCRIPT:")
print(transcript)

Detected language: Hindi


100%|██████████| 804/804 [00:09<00:00, 82.31frames/s]

TRANSCRIPT:
मुजे हाजार उपे चाही दाज दिन में


## 6. Extract amount and deadline

Examples supported:
- `Mere ko 5 din mein 1000 rupaye chahiye`
- `I need 1000 rupees in 5 days`
- `Need ₹2500 within 3 days`

This parser supports common Hindi/English number words up to thousands, but it is not a full language-understanding model. If extraction is wrong, edit the transcript or enter the values manually in the next cell.

In [ ]:
import unicodedata

NUMBER_WORDS = {
    # English
    "zero": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5,
    "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10,
    "eleven": 11, "twelve": 12, "thirteen": 13, "fourteen": 14,
    "fifteen": 15, "sixteen": 16, "seventeen": 17, "eighteen": 18,
    "nineteen": 19, "twenty": 20, "thirty": 30, "forty": 40,
    "fifty": 50, "sixty": 60, "seventy": 70, "eighty": 80, "ninety": 90,
    "hundred": 100, "thousand": 1000,
    # Common Hindi/Hinglish transliterations
    "ek": 1, "do": 2, "teen": 3, "char": 4, "chaar": 4, "paanch": 5,
    "panch": 5, "cheh": 6, "chhe": 6, "saat": 7, "aath": 8, "nau": 9,
    "das": 10, "gyarah": 11, "barah": 12, "terah": 13, "chaudah": 14,
    "pandrah": 15, "solah": 16, "satrah": 17, "atharah": 18, "unnis": 19,
    "bees": 20, "tees": 30, "chalis": 40, "chaalis": 40, "pachas": 50,
    "saath": 60, "sattar": 70, "assi": 80, "nabbe": 90,
    "sau": 100, "hazaar": 1000, "hazar": 1000
}

def words_to_number(phrase):
    tokens = re.findall(r"[a-zA-Z]+|\d[\d,]*", phrase.lower())
    total = 0
    current = 0
    saw_number = False
    for token in tokens:
        if token[0].isdigit():
            value = int(token.replace(",", ""))
            current += value
            saw_number = True
        elif token in NUMBER_WORDS:
            value = NUMBER_WORDS[token]
            saw_number = True
            if value == 100:
                current = max(current, 1) * 100
            elif value == 1000:
                total += max(current, 1) * 1000
                current = 0
            else:
                current += value
        else:
            # Ignore non-number words; continue reading the phrase.
            continue
    return total + current if saw_number else None

def extract_amount_rupees(text):
    t = text.lower().replace("₹", " rupees ").replace("rs.", " rupees ").replace("rs ", " rupees ")
    # Prefer a number/number-word phrase near currency terms.
    currency_pattern = re.compile(
        r"((?:\d[\d,]*(?:\.\d+)?|[a-zA-Z]+)(?:[\s-]+(?:\d[\d,]*(?:\.\d+)?|[a-zA-Z]+)){0,3})\s*"
        r"(?:rupees?|rs\b|inr|rupaye|rupya|rupaiye|रुपये|रुपया)",
        re.IGNORECASE
    )
    m = currency_pattern.search(t)
    if m:
        val = words_to_number(m.group(1))
        if val is not None:
            return float(val)
    # fallback: explicit currency prefix
    m = re.search(r"(?:₹|rs\.?\s*|inr\s*)(\d[\d,]*(?:\.\d+)?)", text, re.IGNORECASE)
    if m:
        return float(m.group(1).replace(",", ""))
    # fallback: words like "need 1000 in 5 days" or "1000 chahiye"
    nums = re.findall(r"\d[\d,]*(?:\.\d+)?", text)
    if nums:
        return float(nums[0].replace(",", ""))
    return None

def extract_deadline_days(text):
    t = text.lower()
    # Numeric deadline patterns
    patterns = [
        r"(?:in|within|after|mein|me|andar|under|within next)\s+(\d[\d,]*)\s*(?:days?|day|din|dino|दिन)",
        r"(\d[\d,]*)\s*(?:days?|day|din|dino|दिन)\s*(?:mein|me|andar|ke andar|within)?",
    ]
    for pattern in patterns:
        m = re.search(pattern, t, re.IGNORECASE)
        if m:
            return int(m.group(1).replace(",", ""))
    # Number-word deadline: scan around day words
    m = re.search(r"\b([a-zA-Z]+(?:[\s-]+[a-zA-Z]+){0,2})\s+(?:days?|day|din|dino)\b", t)
    if m:
        value = words_to_number(m.group(1))
        if value is not None:
            return int(value)
    m = re.search(r"\b(?:in|within|mein|me)\s+([a-zA-Z]+)\s+(?:days?|day|din|dino)\b", t)
    if m:
        value = words_to_number(m.group(1))
        if value is not None:
            return int(value)
    return None

TARGET_RUPEES = extract_amount_rupees(transcript)
DEADLINE_DAYS = extract_deadline_days(transcript)

print("Extracted amount (₹):", TARGET_RUPEES)
print("Extracted deadline (days):", DEADLINE_DAYS)
print("Review the transcript and values. If either is None or wrong, use the manual override cell below.")

Extracted amount (₹): None
Extracted deadline (days): None
Review the transcript and values. If either is None or wrong, use the manual override cell below.


In [ ]:
# OPTIONAL MANUAL OVERRIDE
# Edit these values if the voice transcript was unclear or extraction was wrong.
# Keep None for a value you want to leave unchanged.
MANUAL_AMOUNT_RUPEES = None   # Example: 1000
MANUAL_DEADLINE_DAYS = None   # Example: 5

if MANUAL_AMOUNT_RUPEES is not None:
    TARGET_RUPEES = float(MANUAL_AMOUNT_RUPEES)
if MANUAL_DEADLINE_DAYS is not None:
    DEADLINE_DAYS = int(MANUAL_DEADLINE_DAYS)

if TARGET_RUPEES is None or DEADLINE_DAYS is None:
    print("Amount or deadline could not be confidently extracted.")
    print("Please edit MANUAL_AMOUNT_RUPEES and MANUAL_DEADLINE_DAYS above, then rerun this cell.")
else:
    print(f"Confirmed request: collect ₹{TARGET_RUPEES:,.0f} within {DEADLINE_DAYS} day(s).")

Confirmed request: collect ₹10,000 within 5 day(s).


## 7. Save structured voice input to CSV

In [ ]:
if TARGET_RUPEES is None or DEADLINE_DAYS is None:
    raise ValueError("Set MANUAL_AMOUNT_RUPEES and MANUAL_DEADLINE_DAYS in the previous cell, then rerun it.")

voice_input = pd.DataFrame([{
    "transcript": transcript,
    "target_rupees": float(TARGET_RUPEES),
    "deadline_days": int(DEADLINE_DAYS),
    "asr_model": WHISPER_MODEL_NAME
}])
voice_input.to_csv("/content/voice_request_input.csv", index=False)
display(voice_input)
print("Saved /content/voice_request_input.csv")

,transcript,target_rupees,deadline_days,asr_model
0,"I need 10,000 rupees in 5 days.",10000.0,5,small


Saved /content/voice_request_input.csv


## 8. Upload your Baaki invoice dataset

Upload your existing CSV (for example, `Baaki_large_synthetic_invoice_dataset.csv`). If you do not have it on this machine, use the sample-data cell after this one to test the workflow.

In [ ]:
from google.colab import files
uploaded = files.upload()
csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
if csv_files:
    DATASET_PATH = csv_files[0]
    print("Using:", DATASET_PATH)
else:
    DATASET_PATH = None
    print("No CSV uploaded. Run the sample dataset cell to test.")

Saving Baaki_large_synthetic_invoice_dataset.csv to Baaki_large_synthetic_invoice_dataset (1).csv
Using: Baaki_large_synthetic_invoice_dataset (1).csv


In [ ]:
# OPTIONAL SAMPLE DATASET (only used if you did not upload a CSV)
if DATASET_PATH is None:
    sample = pd.DataFrame([
        {"invoice_id":"INV-1001","customer_id":"C001","customer_name":"Customer A","outstanding_amount_rupees":800,"days_overdue_at_snapshot":8,"customer_on_time_rate_demo":0.90,"expected_payment_days_from_now":2,"disputed":False,"paused":False},
        {"invoice_id":"INV-1002","customer_id":"C002","customer_name":"Customer B","outstanding_amount_rupees":1500,"days_overdue_at_snapshot":20,"customer_on_time_rate_demo":0.70,"expected_payment_days_from_now":4,"disputed":False,"paused":False},
        {"invoice_id":"INV-1003","customer_id":"C003","customer_name":"Customer C","outstanding_amount_rupees":1200,"days_overdue_at_snapshot":5,"customer_on_time_rate_demo":0.95,"expected_payment_days_from_now":1,"disputed":False,"paused":False},
        {"invoice_id":"INV-1004","customer_id":"C004","customer_name":"Customer D","outstanding_amount_rupees":3000,"days_overdue_at_snapshot":35,"customer_on_time_rate_demo":0.40,"expected_payment_days_from_now":10,"disputed":False,"paused":False},
        {"invoice_id":"INV-1005","customer_id":"C005","customer_name":"Customer E","outstanding_amount_rupees":500,"days_overdue_at_snapshot":2,"customer_on_time_rate_demo":0.80,"expected_payment_days_from_now":3,"disputed":True,"paused":False},
    ])
    DATASET_PATH = "/content/baaki_sample_invoices.csv"
    sample.to_csv(DATASET_PATH, index=False)
    print("Created sample dataset:", DATASET_PATH)

## 9. Load and validate invoice dataset

The recommendation code expects these columns:
- `outstanding_amount_rupees`
- `expected_payment_days_from_now`
- `customer_on_time_rate_demo`

Optional columns: `invoice_id`, `customer_id`, `customer_name`, `days_overdue_at_snapshot`, `disputed`, `paused`.

If your real dataset uses different column names, update the mapping in the next cell.

In [ ]:
invoices = pd.read_csv(DATASET_PATH)
print("Dataset rows:", len(invoices))
print("Columns:", invoices.columns.tolist())
display(invoices.head())

Dataset rows: 5000
Columns: ['invoice_id', 'customer_id', 'customer_name', 'invoice_date', 'due_date', 'outstanding_amount_rupees', 'original_amount_rupees', 'days_overdue_at_snapshot', 'customer_on_time_rate_demo', 'paid_by_deadline', 'payment_date', 'disputed', 'paused', 'expected_payment_days_from_now', 'data_type']


,invoice_id,customer_id,customer_name,invoice_date,due_date,outstanding_amount_rupees,original_amount_rupees,days_overdue_at_snapshot,customer_on_time_rate_demo,paid_by_deadline,payment_date,disputed,paused,expected_payment_days_from_now,data_type
0,INV-DEMO-00001,CUST0109,Demo Customer 109,2025-09-26,2025-11-10,6000,6000,4,0.656,0,NaN,0,0,1,SYNTHETIC_DEMO_NOT_REAL
1,INV-DEMO-00002,CUST0127,Demo Customer 127,2025-02-27,2025-04-13,9300,9300,47,0.382,0,NaN,0,0,2,SYNTHETIC_DEMO_NOT_REAL
2,INV-DEMO-00003,CUST0117,Demo Customer 117,2026-03-23,2026-04-22,0,121300,3,0.766,1,2026-04-25,0,0,1,SYNTHETIC_DEMO_NOT_REAL
3,INV-DEMO-00004,CUST0094,Demo Customer 094,2025-07-23,2025-07-30,26200,26200,27,0.759,0,NaN,0,0,5,SYNTHETIC_DEMO_NOT_REAL
4,INV-DEMO-00005,CUST0101,Demo Customer 101,2025-04-10,2025-04-17,0,5300,29,0.980,1,2025-05-16,0,0,2,SYNTHETIC_DEMO_NOT_REAL


## 10. Rank collection candidates
This is an explainable baseline ranking heuristic, not your trained ML model. It filters invoices that are paused/disputed or expected after the deadline, then ranks by expected payment speed and demo on-time rate.

In [ ]:
required = [
    "outstanding_amount_rupees",
    "expected_payment_days_from_now",
    "customer_on_time_rate_demo"
]
missing = [c for c in required if c not in invoices.columns]
if missing:
    raise ValueError(
        f"Dataset is missing required columns: {missing}. "
        "Rename your columns or update the mapping in this cell."
    )

# Fill optional fields with safe defaults if absent.
for col, default in [
    ("invoice_id", ""),
    ("customer_id", ""),
    ("customer_name", ""),
    ("days_overdue_at_snapshot", 0),
    ("disputed", False),
    ("paused", False),
]:
    if col not in invoices.columns:
        invoices[col] = default

for col in ["outstanding_amount_rupees", "expected_payment_days_from_now", "customer_on_time_rate_demo", "days_overdue_at_snapshot"]:
    invoices[col] = pd.to_numeric(invoices[col], errors="coerce")

def as_bool(series):
    return series.astype(str).str.strip().str.lower().isin(["true", "1", "yes", "y"])

eligible = invoices.copy()
eligible = eligible.dropna(subset=[
    "outstanding_amount_rupees",
    "expected_payment_days_from_now",
    "customer_on_time_rate_demo"
])
eligible = eligible[
    (eligible["outstanding_amount_rupees"] > 0) &
    (eligible["expected_payment_days_from_now"] <= int(DEADLINE_DAYS)) &
    (~as_bool(eligible["disputed"])) &
    (~as_bool(eligible["paused"]))
].copy()

# Sort: higher on-time rate first, then sooner expected payment, then older overdue invoice.
eligible = eligible.sort_values(
    by=["customer_on_time_rate_demo", "expected_payment_days_from_now", "days_overdue_at_snapshot"],
    ascending=[False, True, False]
)

# Select invoices until their outstanding total reaches the requested target.
selected_rows = []
running_total = 0.0
for _, row in eligible.iterrows():
    selected_rows.append(row)
    running_total += float(row["outstanding_amount_rupees"])
    if running_total >= float(TARGET_RUPEES):
        break

if selected_rows:
    recommendations = pd.DataFrame(selected_rows).copy()
    recommendations["cumulative_outstanding_rupees"] = recommendations["outstanding_amount_rupees"].cumsum()
    recommendations["within_target_amount"] = recommendations["cumulative_outstanding_rupees"] <= float(TARGET_RUPEES)
else:
    recommendations = eligible.head(0).copy()
    recommendations["cumulative_outstanding_rupees"] = pd.Series(dtype=float)
    recommendations["within_target_amount"] = pd.Series(dtype=bool)

print(f"Target: ₹{TARGET_RUPEES:,.0f} within {DEADLINE_DAYS} day(s)")
print(f"Eligible invoices: {len(eligible)}")
print(f"Recommended invoices: {len(recommendations)}")
print(f"Total recommended outstanding: ₹{recommendations['outstanding_amount_rupees'].sum() if len(recommendations) else 0:,.0f}")
display(recommendations)

Target: ₹10,000 within 5 day(s)
Eligible invoices: 956
Recommended invoices: 1
Total recommended outstanding: ₹106,800


,invoice_id,customer_id,customer_name,invoice_date,due_date,outstanding_amount_rupees,original_amount_rupees,days_overdue_at_snapshot,customer_on_time_rate_demo,paid_by_deadline,payment_date,disputed,paused,expected_payment_days_from_now,data_type,cumulative_outstanding_rupees,within_target_amount
407,INV-DEMO-00408,CUST0103,Demo Customer 103,2026-04-11,2026-04-18,106800,106800,31,0.98,0,NaN,0,0,1,SYNTHETIC_DEMO_NOT_REAL,106800,False


## 11. Export recommendations

In [ ]:
recommendations.to_csv("/content/baaki_voice_collection_recommendations.csv", index=False)
print("Saved:")
print("- /content/voice_request_input.csv")
print("- /content/baaki_voice_collection_recommendations.csv")

from google.colab import files
files.download("/content/voice_request_input.csv")
files.download("/content/baaki_voice_collection_recommendations.csv")

Saved:
- /content/voice_request_input.csv
- /content/baaki_voice_collection_recommendations.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Troubleshooting
- **Microphone permission error:** use Chrome, allow microphone access, and rerun the recording cell.
- **`ffmpeg` not found:** run `!apt-get -qq update && !apt-get -qq install -y ffmpeg`, then rerun recording.
- **Whisper is slow:** set `WHISPER_MODEL_NAME = "base"` and rerun the model-loading cell. Accuracy may be lower.
- **Bad transcript:** try a quieter place, speak slowly, and keep the recording under 15 seconds for testing.
- **Amount/deadline is `None`:** use the manual override cell.
- **Missing dataset columns:** inspect the printed column names and map your CSV's matching columns to the expected names.
- **No eligible invoices:** none match the deadline or all are disputed/paused. Verify the data and deadline.

**Important:** This notebook is a prototype. Verify the transcript, extracted values, invoice status, and recommendation list before acting on them.